In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model
import pickle


In [1]:
## load trained model, scaler and one hot encoder pickle files
model = load_model('model.h5')

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    label_encoder_scaler = pickle.load(file)

with open('ohe_geo.pkl', 'rb') as file:
    ohe_geo = pickle.load(file)

NameError: name 'load_model' is not defined

In [15]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [16]:
##convert input data to dataframe
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [17]:
# convert Gender to numerical value and categorical feature to one hot encoding
encode_geo = ohe_geo.transform([[input_data['Geography']]])
geo_df = pd.DataFrame(encode_geo, columns=ohe_geo.get_feature_names_out(['Geography']))
geo_df

/Users/zarnanakrani/NLP/ANN Classification/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [18]:
input_data = pd.concat([input_df.reset_index(drop=True), geo_df], axis=1)
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [19]:
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [20]:
input_df = pd.concat([input_df.drop("Geography", axis=1), geo_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [21]:
input_scaled = label_encoder_scaler.transform(input_df)
input_scaled

array([[-0.51222865,  0.90636285,  0.10591292, -0.69703024, -0.26147196,
         0.8016426 ,  0.64900815,  0.97238125, -0.86754814,  0.99575899,
        -0.57812007, -0.57330877]])

In [22]:
# predict churn probability
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


array([[0.01431123]], dtype=float32)

In [23]:
prediction_prob = prediction[0][0]
prediction_prob

0.014311233

In [24]:
if prediction_prob >= 0.5:
    print(f'The customer is likely to churn with a probability of {prediction_prob:.2f}')
else:
    print(f'The customer is unlikely to churn with a probability of {prediction_prob:.2f}')

The customer is unlikely to churn with a probability of 0.01
